In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%pip install ../databricks_helpers
dbutils.library.restartPython()

In [0]:
from databricks_helpers.databricks_helper import DatabricksHelpers
dbh = DatabricksHelpers(dbutils, "aviation_analytics",spark)

In [0]:
%sql
--CREATE VOLUME IF NOT EXISTS raw_data;
SELECT current_catalog(), current_schema();

### Putting in streaming data, so that data is not read everytime

[StructuredStreaming](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/index.html)

[AutoLoader/cloudfiles](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)

### Why cloudfiles over spark's direct format
However, Databricks created format("cloudFiles") (Auto Loader) specifically to fix three massive problems with standard Spark streaming.

Here is why cloudFiles is the mandatory best practice for Lakehouse ingestion.

1. The "Directory Listing" Nightmare (Scalability)
When you use standard format("parquet"), Spark relies on basic file system directory listing (like running an ls command in your terminal).

The Problem: If your directory has 1,000 files, it's fast. But if your system has been running for a year and has 5 million Parquet files, standard Spark has to list all 5 million files every single time the stream triggers just to figure out if 10 new files arrived. This takes hours and crashes your cluster.

The cloudFiles Solution: Auto Loader is incredibly smart. It either uses highly optimized RocksDB state tracking to list files instantly, or it hooks directly into your cloud provider's event queue (like AWS SQS, Azure Event Grid, or GCP Pub/Sub). It doesn't list the whole folder; the cloud provider just "pushes" a notification to Databricks saying, "Hey, a new file just landed."

2. Schema Evolution (Resilience)
The Problem: Standard Spark streaming expects a rigid schema. If a new Parquet file arrives with a deleted column or a changed data type, the standard format("parquet") stream will throw an error and completely crash your pipeline in the middle of the night.

The cloudFiles Solution: Auto Loader has built-in schema evolution (schemaEvolutionMode). It detects the change, updates the target Delta table automatically, and keeps the stream running.

3. Cost Efficiency
Because standard Spark takes so long to figure out which files are new, you pay for massive amounts of compute time just for "file discovery." Auto Loader's file discovery is nearly instantaneous, meaning your availableNow=True job finishes faster and costs a fraction of the price.

In [0]:
%sql
CREATE or REFRESH STREAMING TABLE bts_bronze

In [0]:
#dbutils.fs.cp(f"{dbh.volume_directory()}/daily_bts_data/year=2022", f"{bts_streaming_dir}/year=2022", recurse=True)

### Creating bronze layer of flight data

In [0]:
from pyspark.sql.functions import col
import os
import pyspark.pandas as pd

bts_streaming_dir= f"{dbh.volume_directory()}/bts_stream"
os.makedirs(bts_streaming_dir, exist_ok=True)

flight_stream = (spark.readStream
    .format("cloudFiles")#cloudfiles is the format that is specified in databricks (its not in spark documentation), and that is what autoloader all about. 
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{dbh.schemas_directory()}/flights_bronze")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data") # 👈 NEVER lose bad data
    .load(bts_streaming_dir)
    # Add a load timestamp (Best Practice for Bronze)
    .select(
        "*", # Keep all original columns
        col("_metadata.file_modification_time").alias("file_arrival_timestamp"),
        col("_metadata.file_path").alias("source_file_path")
    )
) #it doesnt open any stream as there is no action

(flight_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{dbh.checkpoints_directory()}/flights_bronze") 
    .trigger(availableNow=True)    # it would run only once                
    .option("mergeSchema", "true") #tells to accept new schema
    .table("bronze_flight_data")
)


In [0]:
%sql
select distinct origin from bronze_flight_data;